# loading imports and preprocessing

In [88]:
import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import joblib

In [89]:
df_train = pd.read_csv('dataset/dry_bean.csv')

In [90]:
def eval_metrics(model, X_test, y_true, y_pred):
    accuracy = round(accuracy_score(y_true, y_pred),4)

    # FIX: Add average='weighted' for multi-class support
    precision = round(precision_score(y_true, y_pred, average='weighted'),4)
    recall = round(recall_score(y_true, y_pred, average='weighted'),4)
    f1 = round(f1_score(y_true, y_pred, average='weighted'),4)

    mcc = round(matthews_corrcoef(y_true, y_pred),4)

    # FIX: Use all probabilities and set multi_class='ovr' for AUC
    y_pred_proba = model.predict_proba(X_test)
    auc_score = round(roc_auc_score(y_true, y_pred_proba, multi_class='ovr'),4)
    
    # Return as a dictionary for easier reading later
    return {
        "Accuracy": accuracy,
        "Precision": precision, 
        "Recall": recall, 
        "F1": f1, 
        "MCC": mcc, 
        "AUC": auc_score
    }

In [91]:
X=df_train.drop(columns=['Class'])
y=df_train['Class']
X_train, X_val, y_train, y_val= train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()

# 3. FIT the scaler ONLY on the training data, and transform the training data
X_train_scaled = scaler.fit_transform(X_train)

# 4. Use the EXACT SAME scaler to transform the test data
# CRITICAL: Use .transform(), NOT .fit_transform() here!
X_test_scaled = scaler.transform(X_val)

# logistic regression

In [92]:
lr_model = LogisticRegression(solver='lbfgs', max_iter=1000)

# Train the model
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test_scaled)

logistic_metrics=eval_metrics(lr_model, X_test_scaled, y_val, y_pred_lr)

# Decision Trees

In [93]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_scaled, y_train)
y_pred_dt=dt_model.predict(X_test_scaled)
dt_metrics=eval_metrics(dt_model, X_test_scaled, y_val, y_pred_dt)

# K Nearest Neighbours

In [94]:
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn=knn_model.predict(X_test_scaled)
knn_metrics=eval_metrics(knn_model, X_test_scaled, y_val, y_pred_knn)

# Naive Bayes Classifier

In [95]:
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
y_pred_nb=nb_model.predict(X_test_scaled)
nb_metrics=eval_metrics(nb_model, X_test_scaled, y_val, y_pred_nb)

# SVM

In [96]:
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm=svm_model.predict(X_test_scaled)
svm_metrics=eval_metrics(svm_model, X_test_scaled, y_val, y_pred_svm)

C:\Users\rajul\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [97]:

results_data = [
    {"ML Model Name": "Logistic Regression", **logistic_metrics},
    {"ML Model Name": "Decision Tree", **dt_metrics},
    {"ML Model Name": "KNN", **knn_metrics},
    {"ML Model Name": "Naive Bayes", **nb_metrics},
    {"ML Model Name": "Support Vector Machine", **svm_metrics}
]


results_df = pd.DataFrame(results_data)


columns_order = ["ML Model Name", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
results_df = results_df[columns_order]

results_df = results_df.round(4)

display(results_df)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9266,0.9950,0.9278,0.9266,0.9269,0.9117
1,Decision Tree,0.8928,0.9442,0.8932,0.8928,0.8929,0.8709
2,KNN,0.9192,0.9878,0.9203,0.9192,0.9195,0.9027
3,Naive Bayes,0.9038,0.9919,0.9050,0.9038,0.9039,0.8845
4,Support Vector Machine,0.9339,0.9959,0.9344,0.9339,0.9341,0.9204


dump the models here 

In [98]:
import os

os.makedirs("model", exist_ok=True)

# Save all trained models
joblib.dump(lr_model, "model/logistic_regression.joblib")
joblib.dump(dt_model, "model/decision_tree.joblib")
joblib.dump(knn_model, "model/knn.joblib")
joblib.dump(nb_model, "model/naive_bayes.joblib")
joblib.dump(svm_model, "model/svm.joblib")

joblib.dump(scaler, "model/scaler.joblib")

['model/scaler.joblib']